# Feature Engineering

## Objectives

The objectives of this phase are to:

- Identify the variables that are appropriate for predicting in-hospital mortality.
- Remove identifiers, target variables, and variables that could introduce data leakage.
- Select clinically and analytically relevant demographic, admission, diagnostic and laboratory features.
- Prepare categorical variables for machine learning.
- Determine an appropriate strategy for handling missing values.
- Produce a final modelling dataset suitable for training and evaluating machine learning models.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

pd.set_option("display.max_columns", None)

final_data = pd.read_csv("../data/processed/final_data.csv")

final_data.shape

(129, 36)

In [2]:
final_data.columns

Index(['row_id', 'subject_id', 'hadm_id', 'admittime', 'dischtime',
       'deathtime', 'admission_type', 'admission_location',
       'discharge_location', 'insurance', 'language', 'religion',
       'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis',
       'hospital_expire_flag', 'has_chartevents_data', 'bicarbonate',
       'calcium', 'creatinine', 'glucose', 'magnesium', 'potassium', 'sodium',
       'urea_nitrogen', 'haematocrit', 'haemoglobin', 'platelet_count', 'rdw',
       'white_blood_cells', 'age_at_admission', 'gender',
       'diagnosis_category_model', 'num_diagnoses'],
      dtype='str')

In [3]:
final_data.columns.tolist()

['row_id',
 'subject_id',
 'hadm_id',
 'admittime',
 'dischtime',
 'deathtime',
 'admission_type',
 'admission_location',
 'discharge_location',
 'insurance',
 'language',
 'religion',
 'marital_status',
 'ethnicity',
 'edregtime',
 'edouttime',
 'diagnosis',
 'hospital_expire_flag',
 'has_chartevents_data',
 'bicarbonate',
 'calcium',
 'creatinine',
 'glucose',
 'magnesium',
 'potassium',
 'sodium',
 'urea_nitrogen',
 'haematocrit',
 'haemoglobin',
 'platelet_count',
 'rdw',
 'white_blood_cells',
 'age_at_admission',
 'gender',
 'diagnosis_category_model',
 'num_diagnoses']

## Selecting modelling variables

The modelling dataset is restricted to information that would be available at or within the first 24 hours of hospital admission. Identifiers, post-admission variables, data-availability metadata and variables with limited relevance to the prediction task are excluded.

In [4]:
lab_columns = [
    "bicarbonate",
    "calcium",
    "creatinine",
    "glucose",
    "magnesium",
    "potassium",
    "sodium",
    "urea_nitrogen",
    "haematocrit",
    "haemoglobin",
    "platelet_count",
    "rdw",
    "white_blood_cells"
]
feature_columns = [
    "age_at_admission",
    "gender",
    "diagnosis_category_model",
    "num_diagnoses"
] + lab_columns

model_data = final_data[feature_columns + ["hospital_expire_flag"]].copy()

model_data.shape

(129, 18)

In [5]:
X = model_data[feature_columns]
y = model_data["hospital_expire_flag"]

In [6]:
X.shape

(129, 17)

In [7]:
X.columns.tolist()

['age_at_admission',
 'gender',
 'diagnosis_category_model',
 'num_diagnoses',
 'bicarbonate',
 'calcium',
 'creatinine',
 'glucose',
 'magnesium',
 'potassium',
 'sodium',
 'urea_nitrogen',
 'haematocrit',
 'haemoglobin',
 'platelet_count',
 'rdw',
 'white_blood_cells']

In [8]:
model_data.head()

,age_at_admission,gender,diagnosis_category_model,num_diagnoses,bicarbonate,calcium,creatinine,glucose,magnesium,potassium,sodium,urea_nitrogen,haematocrit,haemoglobin,platelet_count,rdw,white_blood_cells,hospital_expire_flag
0,70.633812,F,Injury_poisoning_other,21,31.0,8.5,3.5,84.0,1.4,4.4,139.0,11.0,36.9,11.7,106.0,19.8,4.6,0
1,36.188912,F,Digestive,6,23.0,8.5,0.7,79.0,1.8,5.9,136.0,3.0,33.9,11.6,299.0,18.9,10.6,1
2,87.082820,F,Infectious,9,29.0,9.0,1.7,134.0,1.3,4.1,138.0,32.0,28.1,8.8,213.0,14.3,13.8,1
3,73.675565,F,Injury_poisoning_other,14,26.0,8.7,0.5,221.0,2.0,4.0,139.0,12.0,30.5,10.9,303.0,12.7,17.8,0
4,48.898015,M,Infectious,14,12.0,8.2,4.0,279.0,1.9,5.2,136.0,31.0,31.5,12.5,85.0,17.2,6.8,1


In [9]:
model_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age_at_admission          129 non-null    float64
 1   gender                    129 non-null    str    
 2   diagnosis_category_model  129 non-null    str    
 3   num_diagnoses             129 non-null    int64  
 4   bicarbonate               126 non-null    float64
 5   calcium                   115 non-null    float64
 6   creatinine                127 non-null    float64
 7   glucose                   126 non-null    float64
 8   magnesium                 119 non-null    float64
 9   potassium                 126 non-null    float64
 10  sodium                    127 non-null    float64
 11  urea_nitrogen             126 non-null    float64
 12  haematocrit               125 non-null    float64
 13  haemoglobin               125 non-null    float64
 14  platelet_count       

## Separating predictors and target

The prediction target is `hospital_expire_flag`, which indicates whether the patient died during the hospital admission. All remaining selected variables are treated as predictors.

The dataset will be divided into predictor variables (`X`) and the target variable (`y`) before model preprocessing.

In [10]:
X = model_data.drop(columns="hospital_expire_flag")
y = model_data["hospital_expire_flag"]

X.shape, y.shape

((129, 17), (129,))

## Train-test split

The dataset is divided into training and test sets before preprocessing or model fitting. The training set will be used to develop the models, while the test set will be reserved for final evaluation.

Stratification is used to preserve a similar proportion of mortality outcomes in both sets.

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [12]:
X_train.shape, X_test.shape

((103, 17), (26, 17))

In [13]:
y_train.value_counts(normalize=True)

hospital_expire_flag
0    0.68932
1    0.31068
Name: proportion, dtype: float64

In [14]:
y_test.value_counts(normalize=True)

hospital_expire_flag
0    0.692308
1    0.307692
Name: proportion, dtype: float64

## Inspecting categorical variables

Categorical variables represent groups or labels rather than continuous numerical measurements. Machine learning models generally require numerical inputs, so these variables will need to be transformed before modelling.

The selected categorical variables are inspected before encoding to understand their categories and identify any sparse groups.

In [15]:
categorical_columns = [
    "gender",
    "diagnosis_category_model"
]

for column in categorical_columns:
    print(f"\n{column}:")
    print(X_train[column].value_counts())


gender:
gender
M    53
F    50
Name: count, dtype: int64

diagnosis_category_model:
diagnosis_category_model
Circulatory               22
Infectious                19
Respiratory               16
Digestive                 16
Injury_poisoning_other    13
Neoplasms                 11
Other                      6
Name: count, dtype: int64


In [16]:
X_train["diagnosis_category_model"].value_counts()

diagnosis_category_model
Circulatory               22
Infectious                19
Respiratory               16
Digestive                 16
Injury_poisoning_other    13
Neoplasms                 11
Other                      6
Name: count, dtype: int64

In [17]:
from sklearn.preprocessing import OneHotEncoder

In [18]:
categorical_features = ["gender", "diagnosis_category_model"]

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore"
)

In [19]:
X_train_encoded = encoder.fit_transform(X_train[categorical_features])

In [20]:
X_train_encoded.shape

(103, 7)

In [21]:
encoder.get_feature_names_out(categorical_features)

array(['gender_M', 'diagnosis_category_model_Digestive',
       'diagnosis_category_model_Infectious',
       'diagnosis_category_model_Injury_poisoning_other',
       'diagnosis_category_model_Neoplasms',
       'diagnosis_category_model_Other',
       'diagnosis_category_model_Respiratory'], dtype=object)

In [22]:
X_test_encoded = encoder.transform(X_test[categorical_features])

In [23]:
X_test_encoded.shape

(26, 7)

In [24]:
X_train[lab_columns].isna().sum()

bicarbonate          2
calcium              9
creatinine           2
glucose              2
magnesium            6
potassium            3
sodium               2
urea_nitrogen        2
haematocrit          2
haemoglobin          2
platelet_count       2
rdw                  2
white_blood_cells    2
dtype: int64

In [25]:
X_test[lab_columns].isna().sum()

bicarbonate          1
calcium              5
creatinine           0
glucose              1
magnesium            4
potassium            0
sodium               0
urea_nitrogen        1
haematocrit          2
haemoglobin          2
platelet_count       2
rdw                  2
white_blood_cells    2
dtype: int64

## Preprocessing

In [26]:
categorical_features = ["gender", "diagnosis_category_model"]

In [27]:
numerical_features = ["age_at_admission"] + lab_columns

In [28]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [29]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [30]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [32]:
X_train.shape, X_test.shape

((103, 17), (26, 17))

In [33]:
y_train.value_counts(normalize=True)

hospital_expire_flag
0    0.68932
1    0.31068
Name: proportion, dtype: float64

In [34]:
y_test.value_counts(normalize=True)

hospital_expire_flag
0    0.692308
1    0.307692
Name: proportion, dtype: float64

In [35]:
X_train_processed = preprocessor.fit_transform(X_train)

In [36]:
X_test_processed = preprocessor.transform(X_test)

In [37]:
X_train_processed.shape, X_test_processed.shape

((103, 21), (26, 21))

In [38]:
preprocessor.get_feature_names_out()

array(['num__age_at_admission', 'num__bicarbonate', 'num__calcium',
       'num__creatinine', 'num__glucose', 'num__magnesium',
       'num__potassium', 'num__sodium', 'num__urea_nitrogen',
       'num__haematocrit', 'num__haemoglobin', 'num__platelet_count',
       'num__rdw', 'num__white_blood_cells', 'cat__gender_M',
       'cat__diagnosis_category_model_Digestive',
       'cat__diagnosis_category_model_Infectious',
       'cat__diagnosis_category_model_Injury_poisoning_other',
       'cat__diagnosis_category_model_Neoplasms',
       'cat__diagnosis_category_model_Other',
       'cat__diagnosis_category_model_Respiratory'], dtype=object)

In [39]:
X_train_processed[:5]

array([[ 0.43264234, -0.08723748,  0.36814946, -0.3594598 , -0.68427918,
         0.99205062,  0.26033517,  0.43320747, -0.00453079, -0.5577753 ,
        -0.60574485,  0.26420511, -0.12162172,  0.23746549,  1.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-0.23312472, -0.28691439, -0.16405257, -0.06792941, -1.06127761,
        -0.3795093 , -0.23017439, -0.09380477, -0.19119921, -1.45160425,
        -1.81414717,  2.68706542, -0.06619235, -0.55607434,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.78615599, -1.68465275, -0.43015358, -0.06792941, -0.93561147,
         3.73517046, -1.53819988, -0.09380477, -0.33120052, -0.4683924 ,
        -0.35134436, -0.70493901, -1.11935038,  0.36545578,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  1.        ,
         0.        ],
       [-0.2903457 ,  0.51179324, -1.49455764, -0.2865772 

In [40]:
print("Training missing values:", np.isnan(X_train_processed).sum())
print("Test missing values:", np.isnan(X_test_processed).sum())

Training missing values: 0
Test missing values: 0


## Preprocessing validation

The preprocessing pipeline produced no missing values in either the training or test datasets following imputation. Numerical features were median-imputed and standardised, while categorical features were one-hot encoded. The resulting datasets contained 22 modelling features.

## Saving processed modelling data

In [41]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out()
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=preprocessor.get_feature_names_out()
)

In [42]:
X_train_processed_df.to_csv("../data/processed/X_train_processed.csv", index=False)
X_test_processed_df.to_csv("../data/processed/X_test_processed.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [43]:
X_train_processed_df.shape

(103, 21)

In [44]:
X_test_processed_df.shape

(26, 21)

In [45]:
preprocessor.get_feature_names_out()

array(['num__age_at_admission', 'num__bicarbonate', 'num__calcium',
       'num__creatinine', 'num__glucose', 'num__magnesium',
       'num__potassium', 'num__sodium', 'num__urea_nitrogen',
       'num__haematocrit', 'num__haemoglobin', 'num__platelet_count',
       'num__rdw', 'num__white_blood_cells', 'cat__gender_M',
       'cat__diagnosis_category_model_Digestive',
       'cat__diagnosis_category_model_Infectious',
       'cat__diagnosis_category_model_Injury_poisoning_other',
       'cat__diagnosis_category_model_Neoplasms',
       'cat__diagnosis_category_model_Other',
       'cat__diagnosis_category_model_Respiratory'], dtype=object)